# LAB  — Unidad 4: El modelo
## Computación en la Nube para IA

**El pedido (PM del piloto de retención):** ya blindamos el modelo para adultos mayores (`SeniorCitizen`), y necesitamos evidencia, justificada, de qué tan bien predecimos churn y a qué costo real de negocio y de equidad.

**Qué hacemos hoy:**
1. Retomamos las 8 columnas de siempre, mismo dataset (`u3_master.telco_churn_raw`)
2. Competencia **XGBoost con y sin mitigaciòn**, cada uno optimizado con **Optuna**. Para cada modelo, comparamos versión **base** vs. **mitigada** (reweighting sobre `SeniorCitizen`)
4. 3 métricas propias, justificadas desde el conocimiento del negocio:
   - **Costo de equidad** (ratio ganancia de equidad / costo en AUC)
   - **Costo de negocio** (USD): FN = $150 (costo de desconexión — truck roll, según benchmarks de industria) · FP = 1 mes de `MonthlyCharges` (oferta de retención regalada innecesariamente)
   - **Score final** para el leaderboard: descalifica modelos con `EOD > 0.2`
5. Todo se registra en **Gemini Enterprise Agent Platform Experiments** — el leaderboard real de la clase
6. Cierre: registramos el mejor modelo en **Model Registry** (sin costo) y dejamos todo listo para el paso a porduccion!


# #superimportante

 gcloud workbench instances create u4-clase --project=computacionnube20261 --location=us-central1-a --machine-type=e2-standard-2 --metadata=idle-timeout-seconds=2700

en shell para crear la intancia correctamente!

In [1]:
# Instalar librerías necesarias
!pip install google-cloud-bigquery google-cloud-aiplatform google-cloud-storage optuna xgboost fairlearn --quiet

print(" Librerías instaladas correctamente")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.6.0 which is incompatible.
 Librerías instaladas correctamente


In [2]:
# ========================================
# CONFIGURACIÓN
# ========================================
from google.cloud import bigquery
from google.cloud import aiplatform
import pandas as pd
import numpy as np


GROUP_NUM = "class"          # formato oficial: g01, g02 ... g08
FECHA_CLASE = "20260904"   # CAMBIA por la fecha real de la clase si corres esto otro día

PROJECT_ID = "computacionnube20261" # CAMBIA por el id correspondiente a tu poryecto
REGION = "us-central1"
SOURCE_TABLE = f"{PROJECT_ID}.u3_master.telco_churn_raw"   

# ---- Nomenclatura oficial: u4_gNN_<tipo>_YYYYMMDD (pautas_entrega_unidad4.docx) ----
BUCKET_NAME = f"{PROJECT_ID}-u4-{GROUP_NUM}-mdl-{FECHA_CLASE}"   
BQ_RESOURCE_NAME = f"u4_{GROUP_NUM}_data_{FECHA_CLASE}"      
EXPERIMENT_NAME = f"u4-{GROUP_NUM}-expchurn-{FECHA_CLASE}"   #guiobajo tiene lios   para el context                          

TARGET = "Churn"
PROTECTED_VAR = "SeniorCitizen"
EOD_MAX = 0.2

COSTO_DESCONEXION_USD = 150
MESES_OFERTA_RETENCION = 1

bq_client = bigquery.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=REGION, experiment=EXPERIMENT_NAME)

print(f"Grupo: {GROUP_NUM} | Fecha: {FECHA_CLASE} | Experimento: {EXPERIMENT_NAME}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Recurso BigQuery: {BQ_RESOURCE_NAME}")
print("Clientes inicializados (BigQuery + Agent Platform)")

Grupo: class | Fecha: 20260904 | Experimento: u4-class-expchurn-20260904
Bucket: computacionnube20261-u4-class-mdl-20260904
Recurso BigQuery: u4_class_data_20260904
Clientes inicializados (BigQuery + Agent Platform)


---
## SECCIÓN 1: Carga y fix rápido

Recuerden: `Churn` llega desde BigQuery como **booleano nativo** (no como string `"Yes"/"No"`) 


In [3]:
# ========================================
# CARGA DE DATOS 
# ========================================

columnas_hoy = [
    'customerID', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'Contract', 'PaymentMethod', 'MonthlyCharges', 'Churn'
]

query = f"SELECT {', '.join(columnas_hoy)} FROM `{SOURCE_TABLE}`"
df = bq_client.query(query).to_dataframe()


df['Churn'] = df['Churn'].astype(int)
df['SeniorCitizen'] = df['SeniorCitizen'].astype(int)

print(f"Filas: {len(df)} | Columnas: {len(df.columns)}")
print(f"\nDistribución de Churn: {df['Churn'].value_counts(normalize=True).round(3).to_dict()}")
df.head(3)


Filas: 7043 | Columnas: 9

Distribución de Churn: {0: 0.735, 1: 0.265}


,customerID,SeniorCitizen,Partner,Dependents,tenure,Contract,PaymentMethod,MonthlyCharges,Churn
0,9426-SXNHE,0,False,False,2,Month-to-month,Bank transfer (automatic),18.75,0
1,3806-YAZOV,0,False,False,3,Month-to-month,Mailed check,18.80,0
2,3387-PLKUI,0,True,True,13,Month-to-month,Mailed check,18.80,0


---
## SECCIÓN 2: Feature engineering + preparación

- One-hot de las categóricas de texto: `Contract`, `PaymentMethod`
- `Partner`, `Dependents` ya son booleanas → cast directo a int (no hace falta one-hot)
- `SeniorCitizen` se queda igual (0/1) — es nuestra variable protegida
- **Sin escalado.** XGBoost es invariante a transformaciones monotónicas por columna: escalar `tenure`/`MonthlyCharges` no le aporta nada al modelo. La versión anterior de esta celda escalaba con `ColumnTransformer` + `StandardScaler`, pero ese paso reordenaba las columnas de salida (las transformadas primero, el resto después) mientras las etiquetaba con el orden *original* — un bug real que desalineaba datos y nombres en las primeras 5 columnas, y que además exigía guardar un scaler que nunca se guardó junto al modelo. Corrección aplicada: se usa `X_train`/`X_test` directamente, sin ninguna transformación adicional. Detalle completo en `PKB_Unidad5_Verificacion_Sesion_20260908.md`, secciones 4 y 5.


automaticos = ['Bank transfer (automatic)', 'Credit card (automatic)']
df['PaymentAutomatic'] = df['PaymentMethod'].isin(automaticos).astype(int)
df

In [4]:
# ========================================
# FEATURE ENGINEERING
# ========================================
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
import re

y = df['Churn']
customer_ids = df['customerID']
X = df.drop(columns=['Churn', 'customerID']).copy()

# Booleanas -> int directo
X['Partner'] = X['Partner'].astype(int)
X['Dependents'] = X['Dependents'].astype(int)

# One-hot de las categóricas de texto
X_encoded = pd.get_dummies(X, columns=['Contract', 'PaymentMethod'], drop_first=True)
#alguntas columnas son Int y no int
X_encoded = X_encoded.astype(float)


def limpiar_nombre_columna(col):
    return re.sub(r'[^0-9a-zA-Z_]', '_', col)

X_encoded.columns = [limpiar_nombre_columna(c) for c in X_encoded.columns]

print(f"Columnas después del encoding: {X_encoded.shape[1]}")
print(list(X_encoded.columns))

# Split -- mismo criterio que Unidad 3
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Guardamos también SeniorCitizen y MonthlyCharges "crudos" del test set (para fairness y costo de negocio)
senior_test = X_test['SeniorCitizen'].reset_index(drop=True)
monthly_test = X_test['MonthlyCharges'].reset_index(drop=True)
customer_ids_train = customer_ids.loc[X_train.index].reset_index(drop=True)
customer_ids_test = customer_ids.loc[X_test.index].reset_index(drop=True)

# ========================================
# SIN ESCALADO -- corrección aplicada (ver PKB_Unidad5_Verificacion_Sesion_20260908.md, secciones 4 y 5)
# ========================================
# XGBoost es invariante a transformaciones monotónicas por columna: escalar no le aporta nada.
# La versión anterior usaba un ColumnTransformer + StandardScaler sobre ['tenure', 'MonthlyCharges'],
# pero ese paso reordena las columnas de salida (transformadas primero, remainder después) mientras
# el código las etiquetaba con el orden ORIGINAL de X_train.columns -- un bug real y silencioso que
# desalineaba datos y nombres en las primeras 5 posiciones, y que además exigía guardar un scaler
# que nunca se guardó junto al modelo (imposible de replicar en inferencia sin él).
#
# Al no escalar, X_train/X_test conservan su orden original y coinciden exactamente con lo que
# dice X_train.columns -- se elimina el bug de raíz, no se parchea.
#
# Se conservan los nombres *_scaled solo por compatibilidad con las celdas siguientes del notebook
# (Secciones 3-6 los referencian) -- ya NO representan datos escalados, son alias directos de X_train/X_test.
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

print(f"\nTrain: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")
print(f"Orden real de columnas (sin escalado, coincide exactamente con las etiquetas): {list(X_train_scaled.columns)}")


Columnas después del encoding: 10
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MonthlyCharges', 'Contract_One_year', 'Contract_Two_year', 'PaymentMethod_Credit_card__automatic_', 'PaymentMethod_Electronic_check', 'PaymentMethod_Mailed_check']

Train: (5634, 10) | Test: (1409, 10)
Orden real de columnas (sin escalado, coincide exactamente con las etiquetas): ['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MonthlyCharges', 'Contract_One_year', 'Contract_Two_year', 'PaymentMethod_Credit_card__automatic_', 'PaymentMethod_Electronic_check', 'PaymentMethod_Mailed_check']


---
## SECCIÓN 3: Mitigación (reweighting)

Igual que en Unidad 3: `sample_weight` calculado por la intersección `SeniorCitizen × Churn`, para que el modelo no aprenda a "ignorar" casos del grupo minoritario dentro de cada clase.

**Decisión de diseño** hacemos dos búsquedas de hiperparámetros con Optuna en paralelo:
* Búsqueda A, con sample_weight incluido desde el inicio (la arquitectura pensada para el objetivo justo)
* Búsqueda B, sin sample_weight (la arquitectura pensada solo para AUC, ciega a la equidad).

Con esas dos arquitecturas, entrenamos las 4 combinaciones posibles, cada una con y sin reweighting,  para poder separar dos efectos distintos: cuánto cuesta la mitigación (comparando con/sin reweighting dentro de la misma arquitectura) y cuánto importa la arquitectura en sí (comparando A vs. B dentro del mismo estado de mitigación). Si el costo de la mitigación sale parecido en A y en B, es evidencia de que es un costo estable, no un artefacto de qué arquitectura se usó,y si sale distinto, es un hallazgo real sobre cómo interactúan ambas decisiones.


<div align="center">
  <h2>Fase de experimentaciòn</h2>
</div>

| ⚖️ Búsqueda A (Regla Justa) | 🎯 Búsqueda B (Regla Ciega) |
| :---: | :---: |
| Optimiza incluyendo `sample_weight` desde el inicio | Optimiza enfocado 100% en AUC, sin pesos |
| ⬇️ | ⬇️ |
| 🏗️ **ARQUITECTURA "A"** | 🏗️ **ARQUITECTURA "B"** |

<br>

<div align="center">
  <h2>Fase   La Matriz Cruzada</h2>
</div>

| Arquitectura Base | ⚖️ ENTRENAMIENTO JUSTO <br> *(Con sample_weight)* | 🎯 ENTRENAMIENTO CIEGO <br> *(Sin sample_weight)* |
| :--- | :--- | :--- |
| 🏗️ **Arquitectura A** | 📦 **Modelo: A-Con** <br> AUC: **0.78** <br>  | 📦 **Modelo: A-Sin** <br> AUC: **0.82** <br>  |
| 🏗️ **Arquitectura B** | 📦 **Modelo: B-Con** <br> AUC: **0.77** <br>  | 📦 **Modelo: B-Sin** <br> AUC: **0.85** <br>  |

<br>

<div align="center">
  <h2>Fase Separación de Efectos</h2>
</div>

**1. EL COSTO DE LA MITIGACIÓN** 
* **En la Arquitectura A:** Bajar de 0.82 a 0.78 = Pierdes **0.04** de AUC por ser justo.
* **En la Arquitectura B:** Bajar de 0.85 a 0.77 = Pierdes **0.08** de AUC por ser justo.

**2. EL IMPACTO DE LA ARQUITECTURA** 
* Si no usas pesos (ciego), **B es mejor** (0.85 vs 0.82).
* Si sí usas pesos (justo), **A es ligeramente mejor** (0.78 vs 0.77).

In [5]:
# ========================================
# SAMPLE WEIGHTS PARA REWEIGHTING (sobre el train set)
# ========================================

grupo_interseccion_train = X_train_scaled['SeniorCitizen'].astype(str) + "_" + y_train.astype(str)
sample_weights_train = compute_sample_weight(class_weight='balanced', y=grupo_interseccion_train)

print("Pesos calculados (primeros 10):")
print(sample_weights_train[:10])


Pesos calculados (primeros 10):
[0.38941111 0.38941111 0.38941111 0.38941111 0.38941111 2.69827586
 0.38941111 0.38941111 0.38941111 0.38941111]


---
## SECCIÓN 4: Competencia entre modelos 

**Reglas de la competencia (idénticas para ambos, para que sea justa):**
- Mismo split, mismo preprocesamiento
- Mismo presupuesto de búsqueda: n_trials=20 por cada búsqueda (A y B), para XGBoost 
- Búsqueda A: optimiza AUC-ROC con validación cruzada de 5 folds, con sample_weight incluido (la arquitectura pensada para el objetivo justo)
- Búsqueda B: optimiza AUC-ROC con validación cruzada de 5 folds, sin sample_weight (la arquitectura pensada solo para AUC, ciega a la equidad)


<div align="center">
  <h2> Para recordar> Validación Cruzada (5-Folds)</h2>
</div>

El dataset se divide en 5 partes iguales. En cada iteración (Fold), el modelo entrena con 4 partes y se evalúa en la parte restante. Al final, todo el dataset fue usado para prueba exactamente una vez.

| Iteración | Bloque 1 | Bloque 2 | Bloque 3 | Bloque 4 | Bloque 5 | Resultado |
| :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **Fold 1** | 🧪 *Test* | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | AUC: 0.81 |
| **Fold 2** | 🏋️ *Train* | 🧪 *Test* | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | AUC: 0.79 |
| **Fold 3** | 🏋️ *Train* | 🏋️ *Train* | 🧪 *Test* | 🏋️ *Train* | 🏋️ *Train* | AUC: 0.83 |
| **Fold 4** | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | 🧪 *Test* | 🏋️ *Train* | AUC: 0.77 |
| **Fold 5** | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | 🏋️ *Train* | 🧪 *Test* | AUC: 0.80 |

<br>

<div align="center">
  <h2>El Análisis: Mean AUC vs. Std AUC</h2>
</div>

Al terminar los 5 Folds, obtenemos 5 valores de AUC. Resumirlos correctamente es vital:

*    **AUC Mean (La Media):** Te dice el **rendimiento promedio** de tu modelo (en el ejemplo arriba, la media es **0.80**).
*    **AUC Std (La Desviación):** Te dice la **estabilidad** de tu modelo. Mide qué tanto rebotan los resultados entre los Folds (en el ejemplo, el Std es aprox **0.02**).

---

###  ¿Por qué es un error elegir un modelo mirando SOLO la media?

Imagina que tienes dos modelos compitiendo en Optuna, y ambos consiguen un AUC promedio de 0.80. Si solo miras la media, pensarás que son iguales. Mira lo que revela el STD:

| Modelo | AUC Mean | AUC Std | Comportamiento en los 5 Folds | Veredicto |
| :--- | :--- | :--- | :--- | :--- |
|  **Modelo A** | **0.80** | **± 0.01** | `[0.79, 0.80, 0.81, 0.80, 0.80]` | ✅ **Estable:**  |
|  **Modelo B** | **0.80** | **± 0.12** | `[0.85, 0.7, 0.88, 0.92, 0.65]` | ❌ **Inestable (Peligroso):** (overfitting). |

> **La Regla de Oro:** 
> Un modelo ganador en producción no es solo el que tiene el AUC más alto, sino el que logra un buen AUC **con el STD más bajo posible**. El STD bajo garantiza que tu modelo generaliza bien, sin importar qué subconjunto de datos se encuentre.

In [6]:
# ========================================
# OPTUNA: XGBoost — Búsqueda A (con sample_weight) y Búsqueda B (sin sample_weight)
# ========================================
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def cv_auc_manual(params, usar_peso):
    """CV manual de 5 folds -- necesario porque cross_val_score no reparte
    sample_weight correctamente por fold cuando se pasa vía fit_params."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    for train_idx, val_idx in skf.split(X_train_scaled, y_train):
        X_tr, X_val = X_train_scaled.iloc[train_idx], X_train_scaled.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        fit_kwargs = {}
        if usar_peso:
            grupo_fold = X_tr['SeniorCitizen'].astype(str) + "_" + y_tr.astype(str)
            fit_kwargs['sample_weight'] = compute_sample_weight(class_weight='balanced', y=grupo_fold)
        modelo = XGBClassifier(**params, eval_metric="auc", random_state=42)
        modelo.fit(X_tr, y_tr, **fit_kwargs)
        aucs.append(roc_auc_score(y_val, modelo.predict_proba(X_val)[:, 1]))
    return np.mean(aucs), np.std(aucs)



def objective_xgb(trial, usar_peso):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    return cv_auc_manual(params, usar_peso)   

print("Optimizando XGBoost -- Búsqueda A (con sample_weight)...")
study_xgb_A = optuna.create_study(directions=["maximize", "minimize"], study_name="xgb_churn_A")
study_xgb_A.optimize(lambda trial: objective_xgb(trial, usar_peso=True), n_trials=20, show_progress_bar=True)
print(f" Trials: {len(study_xgb_A.best_trials)}")
for t in study_xgb_A.best_trials:
    print(f"   AUC={t.values[0]:.4f}  std={t.values[1]:.4f}  params={t.params}")

print("\nOptimizando XGBoost -- Búsqueda B (sin sample_weight)...")
study_xgb_B = optuna.create_study(directions=["maximize", "minimize"], study_name="xgb_churn_B")
study_xgb_B.optimize(lambda trial: objective_xgb(trial, usar_peso=False), n_trials=20, show_progress_bar=True)
print(f" Trials (XGBoost B): {len(study_xgb_B.best_trials)}")
for t in study_xgb_B.best_trials:
    print(f"   AUC={t.values[0]:.4f}  std={t.values[1]:.4f}  params={t.params}")


Optimizando XGBoost -- Búsqueda A (con sample_weight)...


  0%|          | 0/20 [00:00<?, ?it/s]

 Trials: 3
   AUC=0.8370  std=0.0101  params={'max_depth': 3, 'learning_rate': 0.016090191771009336, 'n_estimators': 193, 'subsample': 0.7329686589426252, 'colsample_bytree': 0.9947214684728033}
   AUC=0.8366  std=0.0098  params={'max_depth': 3, 'learning_rate': 0.012855402519912984, 'n_estimators': 191, 'subsample': 0.7113044468218926, 'colsample_bytree': 0.8650188383845345}
   AUC=0.8373  std=0.0108  params={'max_depth': 3, 'learning_rate': 0.022731746393864292, 'n_estimators': 193, 'subsample': 0.700585354144389, 'colsample_bytree': 0.8561843187926983}

Optimizando XGBoost -- Búsqueda B (sin sample_weight)...


  0%|          | 0/20 [00:00<?, ?it/s]

 Trials (XGBoost B): 2
   AUC=0.8402  std=0.0096  params={'max_depth': 6, 'learning_rate': 0.011843174882175497, 'n_estimators': 344, 'subsample': 0.6401227676675144, 'colsample_bytree': 0.6125442841843585}
   AUC=0.8433  std=0.0096  params={'max_depth': 3, 'learning_rate': 0.0325248566731411, 'n_estimators': 201, 'subsample': 0.7575502975313245, 'colsample_bytree': 0.8422641985935272}


In [7]:
def elegir_modelo(study):
    trials = study.best_trials
    aucs = np.array([t.values[0] for t in trials])
    stds = np.array([t.values[1] for t in trials])


    auc_norm = (aucs - aucs.min()) / (aucs.max() - aucs.min() + 1e-9)
    std_norm = (stds - stds.min()) / (stds.max() - stds.min() + 1e-9)

    # Distancia al punto ideal (AUC=1, std=0)
    distancias = np.sqrt((1 - auc_norm)**2 + std_norm**2)
    return trials[np.argmin(distancias)]

In [8]:
trial_xgb_A = elegir_modelo(study_xgb_A)
trial_xgb_B = elegir_modelo(study_xgb_B)

print(f"XGBoost A elegido: AUC={trial_xgb_A.values[0]:.4f} std={trial_xgb_A.values[1]:.4f}")
print(f"  params: {trial_xgb_A.params}")
print(f"\nXGBoost B elegido: AUC={trial_xgb_B.values[0]:.4f} std={trial_xgb_B.values[1]:.4f}")
print(f"  params: {trial_xgb_B.params}")

XGBoost A elegido: AUC=0.8370 std=0.0101
  params: {'max_depth': 3, 'learning_rate': 0.016090191771009336, 'n_estimators': 193, 'subsample': 0.7329686589426252, 'colsample_bytree': 0.9947214684728033}

XGBoost B elegido: AUC=0.8433 std=0.0096
  params: {'max_depth': 3, 'learning_rate': 0.0325248566731411, 'n_estimators': 201, 'subsample': 0.7575502975313245, 'colsample_bytree': 0.8422641985935272}


---
## SECCIÓN 5: Base vs. Mitigado> métricas, costos propios, y registro en Experiments

Para cada familia (XGBoost), entrenamos **base** (sin `sample_weight`) y **mitigada** (con `sample_weight`), con los hiperparámetros que ya encontramos. Calculamos:

- **AUC, Precision, Recall** (poder predictivo)
- **DPD, EOD** sobre `SeniorCitizen` (equidad)
- **Costo de equidad** = `(EOD_base - EOD_mitigado) / (AUC_base - AUC_mitigado)` — cuánta equidad ganamos por cada punto de AUC que "pagamos"
- **Costo de negocio (USD)** = `FN × $150 + Σ(FP → MonthlyCharges × 1 mes)` — sobre la versión **mitigada** de cada modelo
- **Score final** = `AUC` si `EOD ≤ 0.2`, si no, descalificado (`-1`)

Todo se loguea a **Agent Platform Experiments** con `EXPERIMENT_NAME = f"u4-{GROUP_NUM}-expchurn-{FECHA_CLASE}" ` 


<div align="center">
  <h2> Scorecard 360°: Evaluación del Modelo de Churn (Telco)</h2>

</div>

<br>

<div align="center">
  <h3>Fase 1: El Rendimiento Base (Métricas Aisladas)</h3>
</div>

|  1. Poder Predictivo (Métricas del Modelo) |  2. Impacto Negocio (Métricas de Equidad) |
| :--- | :--- |
| **AUC:** Capacidad general de distinguir quién cancelará su plan y quién no. | **DPD (Demographic Parity):** ¿Predecimos cancelación con la misma frecuencia para jóvenes que para *SeniorCitizens*? |
| **Precision:** ¿Qué porcentaje de nuestras promociones de retención fueron a clientes que realmente las necesitaban? | **EOD (Equal Opportunity):** De los que *realmente* se iban a ir, ¿detectamos el mismo porcentaje en jóvenes que en *SeniorCitizens*? |
| **Recall:** ¿Qué porcentaje del churn total estamos logrando identificar a tiempo? | *(El objetivo es que DPD y EOD sean lo más cercanos a 0)* |

<br>

<div align="center">
  <h3>Fase 2: El Análisis de Trade-Offs </h3>
</div>

|  3. El Costo de la Equidad (Trade-off Técnico) | 💵 4. El Costo de Negocio (Impacto en USD) |
| :--- | :--- |
| <br> `(EOD_base - EOD_mitigado) / (AUC_base - AUC_mitigado)` | <br> `(FN × $150) + Σ(FP → MonthlyCharges × 1 mes)` |
| **¿Qué significa?**<br>Mide la eficiencia de tu corrección de sesgo. Te dice **cuánta equidad estás comprando por cada punto de AUC que sacrificas** al aplicar los pesos. | **¿Qué significa?**<br>• **FN (Falsos Negativos):** El cliente se fugó y no lo vimos. Cuesta **$150** (costo de adquirir un cliente nuevo).<br>• **FP (Falsos Positivos):** Creímos que se iba y le dimos una promo innecesaria. Cuesta **1 mes de su tarifa real** (`MonthlyCharges`). |

<br>

<div align="center">
  <h3>Fase 3: El Veredicto Final (Regla de Selección)</h3>
</div>

Para elegir el mejor modelo, el negocio impone una regla estricta donde la ética funciona como filtro, y el AUC como métrica de desempate.

| Condición del Modelo | 🏆 Score Final Asignado | Interpretación |
| :--- | :---: | :--- |
| 🟢 **EOD ≤ 0.2** *(Justo)* | **El valor del AUC** (Ej. `0.82`) | El modelo pasa la auditoría de equidad para *SeniorCitizens*. Compite normalmente por ser el mejor. |
| 🔴 **EOD > 0.2** *(Injusto)* | **-1** *(Descalificado)* | El modelo discrimina demasiado al grupo minoritario. No importa si su AUC es de 0.99, no puede ir a producción. |

In [9]:
# ========================================
# SECCIÓN 5: Entrenar las 4 configuraciones
# ========================================
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

def entrenar_y_evaluar(nombre_config, model_class, params, sample_weight=None):
    modelo = model_class(**params)
    fit_kwargs = {}
    if sample_weight is not None:
        fit_kwargs['sample_weight'] = sample_weight

    modelo.fit(X_train_scaled, y_train, **fit_kwargs)

    y_pred = modelo.predict(X_test_scaled)
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=senior_test)
    eod = equalized_odds_difference(y_test, y_pred, sensitive_features=senior_test)
    cm = confusion_matrix(y_test, y_pred)

    return {
        "modelo": modelo, "auc": auc, "precision": prec, "recall": rec,
        "dpd": dpd, "eod": eod, "confusion_matrix": cm, "y_pred": y_pred
    }

# Hiperparámetros de las 2 arquitecturas ya elegidas
xgb_params_A = {**trial_xgb_A.params, "eval_metric": "auc", "random_state": 42}
xgb_params_B = {**trial_xgb_B.params, "eval_metric": "auc", "random_state": 42}

print("Entrenando las 4 configuraciones")

resultados = {}
resultados["A_base"]     = entrenar_y_evaluar("A_base", XGBClassifier, xgb_params_A, sample_weight=None)
resultados["A_mitigado"] = entrenar_y_evaluar("A_mitigado", XGBClassifier, xgb_params_A, sample_weight=sample_weights_train)
resultados["B_base"]     = entrenar_y_evaluar("B_base", XGBClassifier, xgb_params_B, sample_weight=None)
resultados["B_mitigado"] = entrenar_y_evaluar("B_mitigado", XGBClassifier, xgb_params_B, sample_weight=sample_weights_train)

print("Las 4 configuraciones entrenadas y evaluadas")

Entrenando las 4 configuraciones
Las 4 configuraciones entrenadas y evaluadas


In [10]:
# ========================================
# MÉTRICAS PROPIAS: costo de equidad, costo de negocio, score final
# ========================================
def costo_equidad_ratio(res_base, res_mit):
    delta_auc = res_base["auc"] - res_mit["auc"]
    delta_eod = res_base["eod"] - res_mit["eod"]
    if delta_auc <= 0:
        return None
    return round(delta_eod / delta_auc, 4)

def costo_negocio_usd(res, monthly_charges_test):
    y_pred = res["y_pred"]
    y_true = y_test.reset_index(drop=True)
    y_pred_s = pd.Series(y_pred).reset_index(drop=True)
    falsos_negativos = (y_true == 1) & (y_pred_s == 0)
    falsos_positivos = (y_true == 0) & (y_pred_s == 1)
    costo_fn = falsos_negativos.sum() * COSTO_DESCONEXION_USD
    costo_fp = (monthly_charges_test[falsos_positivos] * MESES_OFERTA_RETENCION).sum()
    return round(costo_fn + costo_fp, 2)

def score_final(res, eod_max=EOD_MAX):
    return round(res["auc"], 4) if res["eod"] <= eod_max else -1


metricas_finales = []
for arquitectura, res_key in [("Arquitectura A", "A_mitigado"), ("Arquitectura B", "B_mitigado")]:
    res_mit = resultados[res_key]
    res_base = resultados[res_key.replace("_mitigado", "_base")]   # "A_mitigado" -> "A_base"

    fila = {
        "Arquitectura": arquitectura,
        "AUC": round(res_mit["auc"], 4),
        "Precision": round(res_mit["precision"], 4),
        "Recall": round(res_mit["recall"], 4),
        "DPD": round(res_mit["dpd"], 4),
        "EOD": round(res_mit["eod"], 4),
        "AUC_base (sin mitigar)": round(res_base["auc"], 4),
        "EOD_base (sin mitigar)": round(res_base["eod"], 4),
        "Costo_equidad_ratio": costo_equidad_ratio(res_base, res_mit),
        "Costo_negocio_USD": costo_negocio_usd(res_mit, monthly_test),
    }
    metricas_finales.append(fila)

df_leaderboard = pd.DataFrame(metricas_finales).sort_values("Recall", ascending=False)
print("=" * 90)
print("LEADERBOARD (Arquitectura A vs. B, ambas mitigadas)")
print("=" * 90)
print(df_leaderboard.to_string(index=False))

ganador = df_leaderboard.iloc[0]["Arquitectura"]
if df_leaderboard.iloc[0]["EOD"] <= EOD_MAX:
    print(f"\n Ganadora: {ganador} (mayor Recall, y pasa el filtro de equidad EOD ≤ {EOD_MAX})")
else:
    print(f"\n {ganador} tiene el mayor Recall, pero NO pasa el filtro de equidad (EOD ≤ {EOD_MAX})")


LEADERBOARD (Arquitectura A vs. B, ambas mitigadas)
  Arquitectura    AUC  Precision  Recall    DPD    EOD  AUC_base (sin mitigar)  EOD_base (sin mitigar)  Costo_equidad_ratio  Costo_negocio_USD
Arquitectura B 0.8294     0.5061  0.7781 0.1484 0.1007                  0.8339                  0.3013               45.234            34074.9
Arquitectura A 0.8340     0.5244  0.7754 0.1673 0.1082                  0.8353                  0.2828              135.183            32194.2

 Ganadora: Arquitectura B (mayor Recall, y pasa el filtro de equidad EOD ≤ 0.2)


In [11]:
# ========================================
# REGISTRO EN GEMINI ENTERPRISE AGENT PLATFORM EXPERIMENTS
# ========================================
from sklearn.metrics import roc_curve

xgb_params_por_arquitectura = {"A": xgb_params_A, "B": xgb_params_B}

for arquitectura, res_key in [("Arquitectura A", "A_mitigado"), ("Arquitectura B", "B_mitigado")]:
    letra = arquitectura.replace("Arquitectura ", "")   

    res_mit = resultados[res_key]
    res_base = resultados[f"{letra}_base"]
    fila = next(f for f in metricas_finales if f["Arquitectura"] == arquitectura)

    run_name = f"u4-{GROUP_NUM}-exp-{FECHA_CLASE}-{letra.lower()}"

    try:
        run_ctx = aiplatform.start_run(run_name, resume=True) #reabre un run anterior
    except Exception:
        run_ctx = aiplatform.start_run(run_name) #crea un run

    with run_ctx as run:
        params_usados = xgb_params_por_arquitectura[letra]
        run.log_params({k: str(v) for k, v in params_usados.items()})
        run.log_params({"protected_var": PROTECTED_VAR, "eod_max": EOD_MAX, "group_num": GROUP_NUM})

        run.log_metrics({
            "auc": fila["AUC"], "precision": fila["Precision"], "recall": fila["Recall"],
            "dpd": fila["DPD"], "eod": fila["EOD"],
            "auc_base": fila["AUC_base (sin mitigar)"], "eod_base": fila["EOD_base (sin mitigar)"],
            "costo_negocio_usd": fila["Costo_negocio_USD"],
        })
        if fila["Costo_equidad_ratio"] is not None:
            run.log_metrics({"costo_equidad_ratio": fila["Costo_equidad_ratio"]})

        fpr, tpr, thresh = roc_curve(y_test, res_mit["modelo"].predict_proba(X_test_scaled)[:, 1])
        thresh = np.clip(thresh, 0, 1)
        aiplatform.log_classification_metrics(
            labels=["No Churn", "Churn"],
            matrix=res_mit["confusion_matrix"].tolist(),
            fpr=fpr.tolist(), tpr=tpr.tolist(), threshold=thresh.tolist(),
            display_name=f"cm_{run_name}",
        )

    print(f"Logueado a Experiments: {run_name}")



Logueado a Experiments: u4-class-exp-20260904-a


Logueado a Experiments: u4-class-exp-20260904-b


---
## SECCIÓN 6: Registro en Model Registry 

Guardamos los 2 modelos (Arquitectura A y Arquitectura B, ambos mitigados) en Cloud Storage y los registramos en Model Registry, el que tuvo mayor Recall (y pasó el filtro de equidad) queda marcado como ganador vía labels, pero ambos se conservan como evidencia. No creamos Endpoint ni desplegamos, eso queda para Unidad 6 .

* Nota de actualidad: existe Agent Platform Feature Store para gestión de features reutilizables, pero su modo de online serving está deprecado desde mayo 2026 (apagado total en febrero 2027) — Google recomienda BigQuery/Bigtable directo. Para este ejercicio no lo necesitamos: los datos ya viven en BigQuery.


In [12]:
# ========================================
# GUARDAR Y REGISTRAR LOS 2 MODELOS (A y B) — sin deploy
# ========================================
import joblib, os
from google.cloud import storage

# Verifica tu versión de xgboost antes de elegir el contenedor
!pip show xgboost | grep Version

BUCKET_NAME = f"{PROJECT_ID}-u4-{GROUP_NUM}-mdl-{FECHA_CLASE}"
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    print(f"El bucket {BUCKET_NAME} no existe -- creándolo en {REGION}...")
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"✅ Bucket creado: {BUCKET_NAME}")
else:
    print(f"✅ Bucket ya existe: {BUCKET_NAME}")


# NOTA (sesión 2026-09-08): el contenedor prediseñado tope subió de 1-4 a 2-1, pero sigue sin
# alcanzar la versión de xgboost usada aquí para entrenar -- verificar con
# "pip show xgboost | grep Version" (celda de abajo) antes de decidir. Ver
# PKB_Unidad5_Verificacion_Sesion_20260908.md sección 3 para la evidencia completa.
SERVING_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-4:latest"  # <- revisar antes de correr

modelos_uploaded = {}

for letra in ["A", "B"]:
    modelo_arq = resultados[f"{letra}_mitigado"]["modelo"]

    # No hace falta Pipeline ni scaler -- el modelo ya no depende de ninguna transformación
    # externa (ver corrección en Sección 2). Guardamos solo el XGBClassifier.
    MODEL_DIR = f"modelo_churn_{letra}" #tengan cuidado con la nomenclatura!
    os.makedirs(MODEL_DIR, exist_ok=True)
    joblib.dump(modelo_arq, f"{MODEL_DIR}/model.joblib")

    GCS_PREFIX = f"modelo_churn_{letra}"
    GCS_MODEL_PATH = f"gs://{BUCKET_NAME}/{GCS_PREFIX}"

    blob = bucket.blob(f"{GCS_PREFIX}/model.joblib")
    blob.upload_from_filename(f"{MODEL_DIR}/model.joblib")
    print(f"✅ Modelo {letra} subido a {GCS_MODEL_PATH}/model.joblib")

    es_ganador = (letra == ganador.replace("Arquitectura ", ""))

    model_uploaded = aiplatform.Model.upload(
        display_name=f"u4-{GROUP_NUM}-mdl-{FECHA_CLASE}-{letra.lower()}",
        artifact_uri=GCS_MODEL_PATH,
        serving_container_image_uri=SERVING_CONTAINER,
        labels={"ganador": "true" if es_ganador else "false", "arquitectura": letra.lower()},
    )

    modelos_uploaded[letra] = model_uploaded
    print(f"✅ Modelo {letra} registrado (SIN desplegar){' -- GANADOR' if es_ganador else ''}: {model_uploaded.resource_name}")



Version: 3.4.1
El bucket computacionnube20261-u4-class-mdl-20260904 no existe -- creándolo en us-central1...
✅ Bucket creado: computacionnube20261-u4-class-mdl-20260904
✅ Modelo A subido a gs://computacionnube20261-u4-class-mdl-20260904/modelo_churn_A/model.joblib
Creating Model
Create Model backing LRO: projects/820873399990/locations/us-central1/models/8863726731211571200/operations/3659223990578184192
Model created. Resource name: projects/820873399990/locations/us-central1/models/8863726731211571200@1
To use this Model in another session:
model = aiplatform.Model('projects/820873399990/locations/us-central1/models/8863726731211571200@1')
✅ Modelo A registrado (SIN desplegar): projects/820873399990/locations/us-central1/models/8863726731211571200
✅ Modelo B subido a gs://computacionnube20261-u4-class-mdl-20260904/modelo_churn_B/model.joblib
Creating Model
Create Model backing LRO: projects/820873399990/locations/us-central1/models/4406289000021622784/operations/6539838902235037696
M

In [ ]:
#borrar todo despues

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project="computacionnube20261", location="us-central1", experiment="churn-unidad4")
for run in aiplatform.ExperimentRun.list(experiment="churn-unidad4"):
    print(run.name)
    run.delete()

In [ ]:
models = aiplatform.Model.list(filter='display_name="modelo_churn_unidad4_clase"')
for m in models:
    m.delete()

In [ ]:
from google.cloud import storage
client = storage.Client(project="computacionnube20261")
bucket = client.bucket("computacionnube20261-clase-models")
if bucket.exists():
    bucket.delete(force=True)  # force=True borra el contenido también

In [ ]:
bq_client.delete_table("computacionnube20261.u3_master.clientes_nuevos_clase", not_found_ok=True)